# In-memory HEFT operators

Start from white noise, evolve particles with pmwd, and paint five hybrid effective field theory (HEFT) operators in memory. The cosmology, simulation settings, and random seed follow the [pmwd quickstart](quickstart.ipynb).

From the repository root, run `python -m pip install '.[vis]' ipykernel` in the Python environment you will select as the Jupyter kernel. For GPU execution, use an environment with GPU-enabled JAX. The first cell shows the backend and visible devices.


In [ ]:
import jax
import matplotlib.pyplot as plt
import numpy as np

from pmwd import Configuration, Cosmology, white_noise
from pmwd.heft import simulate_heft

print('JAX backend:', jax.default_backend())
print('JAX devices:', jax.devices())


## Configuration and initial conditions

Both quickstart configurations use a periodic box of side $256\,h^{-1}\mathrm{Mpc}$. On GPU, the particle grid is $256^3$ with spacing $1\,h^{-1}\mathrm{Mpc}$. On CPU, it is $64^3$ with spacing $4\,h^{-1}\mathrm{Mpc}$. The force mesh has twice as many cells per axis as the particle grid. For the HEFT output, we choose one cell per Lagrangian particle-grid cell.

The pmwd defaults use double precision for cosmological parameters and single precision for particle displacements and painted fields. All five volume means vanish analytically. The printed means show the residuals from finite-precision arithmetic.


In [ ]:
if jax.default_backend() == 'gpu':
    ptcl_spacing = 1.
    ptcl_grid_shape = (256,) * 3
else:
    ptcl_spacing = 4.
    ptcl_grid_shape = (64,) * 3

conf = Configuration(ptcl_spacing, ptcl_grid_shape, mesh_shape=2)
output_grid = conf.ptcl_grid_shape[0]
print(f'{conf.ptcl_num:,} particles, force mesh {conf.mesh_shape}, '
      f'{conf.a_nbody_num} N-body steps, HEFT grid {(output_grid,) * 3}')


The native linear-theory path uses pmwd's Eisenstein–Hu transfer fit and integrated growth, followed by 2LPT and N-body evolution. No external transfer files are needed.

Generate white noise outside any differentiated function. Keep these amplitudes fixed when varying cosmological parameters; a new random seed would change the realization as well as the cosmology.


In [ ]:
cosmo = Cosmology(conf, A_s_1e9=2.0, n_s=0.96, Omega_m=0.3,
                  Omega_b=0.05, h=0.7)
seed = 0
modes = white_noise(seed, conf)


## Construct all five fields

The same **unsmoothed** native linear modes feed both gravity and the HEFT branch. Only the HEFT weights receive the Gaussian field-amplitude window $\exp[-k^2/(2k_d^2)]$, with $k_d=0.75\,h\,\mathrm{Mpc}^{-1}$ here. The alternative `smoothing='csst'` uses the radial field window $\exp[-R^2\max(k-k_\mathrm{pivot},0)^2/4]$, with defaults $R=5\,h^{-1}\mathrm{Mpc}$ and $k_\mathrm{pivot}=1\,h\,\mathrm{Mpc}^{-1}$.

Weights refer to `a_ref=1` (redshift zero); evolution ends at `conf.a_stop`. These are separate choices. The reference epoch must lie in the native growth-table range `(0, conf.a_stop]`, so the default reference requires `a_stop >= 1`.


In [ ]:
operators = simulate_heft(
    modes, cosmo, conf, a_ref=1., output_grid=output_grid,
    smoothing='gaussian', kd=0.75,
)
operators = jax.block_until_ready(operators)


Let $A[w]$ be weighted CIC advection with one normalization by output-cell count divided by particle count. The returned channel names are `O1`, `Od`, `OL`, `Od2`, and `Os2`, in the order

$$[A[1]-1,\ A[d],\ A[\nabla^2 d],\ A[Q],\ A[S]],$$

where $d$ is smoothed linear density, $Q=d^2-\langle d^2\rangle_q$, and $S=\sum_{ij}K_{ij}^2-\langle\sum_{ij}K_{ij}^2\rangle_q$ for the traceless tidal tensor $K_{ij}$. Quadratic products use spectral padding and projection, including Nyquist splitting/merging, before advection. No local density division or second trace subtraction is applied.

Output nodes lie at cell centers, offset by half an output cell from the box origin. All channels except the Laplacian are dimensionless; `OL` has units $(h/\mathrm{Mpc})^2$ with the default length unit.


In [ ]:
labels = ['O1: matter overdensity', 'Od: linear density', 'OL: Laplacian',
          'Od2: centered density squared', 'Os2: centered tidal squared']
fields = np.asarray(operators)
assert fields.shape == (5, output_grid, output_grid, output_grid)
assert fields.dtype == conf.float_dtype
print('shape:', fields.shape, 'dtype:', fields.dtype)
for label, field in zip(labels, fields):
    assert np.isfinite(field).all()
    mean = field.mean(dtype=np.float64)
    rms = np.sqrt(np.mean(field**2, dtype=np.float64))
    if field.dtype == np.float64:
        assert abs(mean) <= 2e-11
    print(f'{label:29s} mean={mean: .3e}  rms={rms:.3e}')


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.5), layout='constrained')
box_size = conf.box_size[0]
for ax, field, label in zip(axes, fields, labels):
    section = field[:, :, output_grid // 2].T
    limit = np.max(np.abs(section))
    picture = ax.imshow(section, origin='lower', extent=(0, box_size, 0, box_size),
                        cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel(r'$x\ [h^{-1}\mathrm{Mpc}]$')
    fig.colorbar(picture, ax=ax, shrink=0.7)
axes[0].set_ylabel(r'$y\ [h^{-1}\mathrm{Mpc}]$')
plt.show()


## Reuse existing in-memory particles

For an existing simulation, import `heft_operators` and `linear_field_from_modes` from `pmwd.heft`. The lower-level input is a cubic complex rFFT with `norm='forward'` of linear density at the declared reference epoch. Native `linear_modes(..., a=None)` instead uses physical-volume coefficients: the conversion is `native_modes * growth(a_ref) / conf.box_vol`. Growth at `a=1` is **not** assumed to be one. If the modes already contain growth at an epoch, supply `modes_a` to the adapter to avoid applying it twice.

The evolved `Particles` must retain the original C-order Lagrangian row association (z fastest), as native LPT/N-body does. Particle position is `pmid * conf.cell_size + disp`, not displacement alone. Restore row order after any sorting. Low-level field and displacement precision must match `conf.float_dtype`.

The path supports reverse-mode gradients in white-noise amplitudes and continuous cosmological parameters, including both weight and trajectory changes. Keep configuration and HEFT options static when compiling with JAX. Native evolution compiles its own steps; a monolithic JIT is unnecessary. Forward-mode JVP is not promised by native custom-VJP operations.


## Optional final-output save

The result is already available as `operators`. Enable the following cell only to save final fields and their labels. Exclusive creation prevents overwriting an existing result; intermediate arrays are never saved by the HEFT module.


In [ ]:
save_final = False
if save_final:
    from pathlib import Path
    destination = Path('heft_operators.npz')
    with destination.open('xb') as stream:
        np.savez_compressed(stream, operators=fields, labels=np.asarray(labels),
                            a_ref=1., a_stop=conf.a_stop, box_size=conf.box_size[0])
    print(f'Saved {destination}')
